In [1]:
import sys
sys.path.append("..")

In [2]:
import tqdm
import os
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import cvxpy as cp
from copy import deepcopy

from src.data import *
from src.model import *
from src.recourse import *
from src.utils import *

warnings.filterwarnings('ignore')

In [3]:
def append_result(d, algorithm, seed, alpha, lamb, i, x_0, theta_0, x_r, theta_r=None):
    d["algorithm"].append(algorithm)
    d["seed"].append(seed)
    d["alpha"].append(alpha)
    d["lambda"].append(lamb)
    d["i"].append(i)
    d["x_0"].append(x_0.round(4))
    d["x_r"].append(x_r.round(4))
    d["theta_0"].append(theta_0.round(4))

In [4]:
def recourse_runner(seed: int, X: np.ndarray, recourse: Recourse, params: dict, dataset: Dataset):
    alpha = params['alpha']
    lamb = params['lamb']
    recourse_i = params['recourse_index']
    f_name = f'../results/recourse/lr_{dataset.name}_{recourse.name}_{lamb}_{alpha}_{seed}.pkl'
    
    results = {'algorithm': [], 'seed': [], 'alpha': [], 'lambda': [], 'i': [], 'x_0': [], 'x_r': [], 'theta_0': []}
    weights_0, bias_0 = recourse.weights, recourse.bias
    theta_0 = np.hstack((weights_0, bias_0))
    if recourse.name == "ROAR":
        print(weights_0, bias_0, theta_0)
    n = len(X)

    for i in tqdm.trange(n, desc=f'[{recourse.name}] [alpha={alpha}] [lambda={lamb}]', colour='#0091ff'):
        x_0 = X[i]
        x_r = recourse.get_recourse(x_0)
        append_result(results, recourse.name, seed, alpha, lamb, recourse_i[i], x_0, theta_0, x_r)

    df_results = pd.DataFrame(results)
    if params['append_results'] and os.path.exists(f_name):
        df_tmp = pd.read_pickle(f_name)
        df_results = pd.concat((df_tmp, df_results), axis=0).sort_values(['i'], ignore_index=True)

    if params["save_results"]:
        print(f'[{recourse.name}] Saving results for {dataset.name} run {seed}')
        df_results.to_pickle(f_name)
    
    return df_results

In [5]:
def run_experiment(dataset: Dataset, recourse_fns: List[Recourse], params: dict, results: List):
    alpha = params['alpha']
    lamb = params['lamb']
    
    for seed in params['seeds']:
        train_data, test_data = dataset.get_data(seed)
        X_train, y_train = train_data
        X_test, y_test = test_data
        
        base_model = LR()
        base_model.train(X_train.values, y_train.values)
        
        weights_0 = base_model.model.coef_[0]
        bias_0 = base_model.model.intercept_
        
        recourse_needed_X_train = recourse_needed(base_model.predict, X_train.values)
        recourse_needed_X_test = recourse_needed(base_model.predict, X_test.values)
        
        for recourse_fn in recourse_fns:
            recourse_needed_X_test_idx = np.arange(recourse_needed_X_test.shape[0])

            recourse = recourse_fn(weights=weights_0, bias=bias_0, alpha=alpha, lamb=lamb)
            if params["lamb"] is None:
                params['lamb'] = recourse.choose_lambda(recourse_needed_X_train, base_model.predict, X_train.values)
                recourse.lamb = params['lamb']
            
            if params['append_results']:
                f_name = f"../results/recourse/lr_{dataset.name}_{recourse.name}_{lamb}_{alpha}_{seed}.pkl"
                if os.path.exists(f_name):
                    df_tmp = pd.read_pickle(f_name)
                    recourse_needed_X_test_idx = np.setdiff1d(recourse_needed_X_test_idx, df_tmp['i'].to_numpy())
                else:
                    print(f"{f_name} does not exist. The recourses will be created in a new file")

                if recourse_needed_X_test_idx.size == 0:
                    print(f"{f_name} already has all the recourses. Skipping")
                    continue
            
            if params['subsample']:
                rng = np.random.default_rng(seed=seed)
                size_N = int(np.rint(params['subsample_size'] * recourse_needed_X_test.shape[0]))
                if recourse_needed_X_test_idx.shape[0] < size_N:
                    size_N = recourse_needed_X_test_idx.shape[0]
                recourse_needed_X_test_idx = rng.choice(recourse_needed_X_test_idx, size=size_N, replace=False)
            
            params['recourse_index'] = recourse_needed_X_test_idx.copy()
            recourse_needed_X_test = recourse_needed_X_test[recourse_needed_X_test_idx]
                
            df_results = recourse_runner(seed, recourse_needed_X_test, recourse, params, dataset)
            results.append(df_results)

0 - [24 19 31 0 2 1] |
1- [17 16 27 29 34 4] |
2 - [10 4 31 30 3 18] |
3 - [2 6 27 30 21 26] |
4 - [26 33 34 35 2 36]

In [6]:
np.hstack((np.arange(0.001, 0.0105, 0.001), np.arange(0.02, 0.105, 0.01), np.arange(0.2, 0.55, 0.1))).round(5)

array([0.001, 0.002, 0.003, 0.004, 0.005, 0.006, 0.007, 0.008, 0.009,
       0.01 , 0.02 , 0.03 , 0.04 , 0.05 , 0.06 , 0.07 , 0.08 , 0.09 ,
       0.1  , 0.2  , 0.3  , 0.4  , 0.5  ])

In [ ]:
alphas = [0.1] # <------------------------
lambdas = [0.5, 0.3, 0.1, 0.04, 0.01, 0.004, 0.001] # <------------------------

torch.manual_seed(0)

for lamb in lambdas:
    for alpha in alphas:

        d_results = {}
        params = {}
        params['alpha'] = alpha # float, None
        params['lamb'] = lamb
        params['seeds'] = range(5)
        params['save_results'] = True
        params['append_results'] = False
        params['subsample'] = True
        params['subsample_size'] = 0.075

        datasets = [GermanDataset()] # <------------------------
        recourse_fns = [L1Recourse] # <------------------------

        for dataset in datasets:
            results = []
            print(f'Running {dataset.name} data...')
            run_experiment(dataset, recourse_fns, params, results)
            
            d_results[dataset.name] = pd.concat(results)
            print(f'Finished {dataset.name}\n')

Running german data...


[L1PSD] [alpha=0.1] [lambda=0.5]: 100%|██████████| 4/4 [13:17<00:00, 199.36s/it]


[L1PSD] Saving results for german run 0


[L1PSD] [alpha=0.1] [lambda=0.5]: 100%|██████████| 3/3 [09:50<00:00, 196.72s/it]


[L1PSD] Saving results for german run 1


...

[L1PSD] Saving results for german run 2


[L1PSD] [alpha=0.1] [lambda=0.001]: 100%|██████████| 4/4 [12:07<00:00, 181.91s/it]


[L1PSD] Saving results for german run 3


[L1PSD] [alpha=0.1] [lambda=0.001]: 100%|██████████| 4/4 [11:15<00:00, 168.76s/it]

[L1PSD] Saving results for german run 4
Finished german



In [9]:
alphas = [0.1] # <------------------------
# lambdas = np.hstack((np.arange(0.001, 0.0105, 0.001), np.arange(0.02, 0.105, 0.01), np.arange(0.2, 0.55, 0.1))).round(5) # <------------------------
lambdas = [0.0001, 0.00001] # <------------------------

torch.manual_seed(0)

for lamb in lambdas:
    for alpha in alphas:

        d_results = {}
        params = {}
        params['alpha'] = alpha # float, None
        params['lamb'] = lamb
        params['seeds'] = range(5)
        params['save_results'] = True
        params['append_results'] = False
        params['subsample'] = False
        params['subsample_size'] = 0.25

        datasets = [GermanDataset()] # <------------------------
        recourse_fns = [ROARLInf, ROARL1] # <------------------------

        for dataset in datasets:
            results = []
            print(f'Running {dataset.name} data...')
            run_experiment(dataset, recourse_fns, params, results)
            
            d_results[dataset.name] = pd.concat(results)
            print(f'Finished {dataset.name}\n')

Running german data...


[ROARLInf] [alpha=0.1] [lambda=0.0001]: 100%|██████████| 16/16 [00:13<00:00,  1.17it/s]


[ROARLInf] Saving results for german run 0


[ROARL1] [alpha=0.1] [lambda=0.0001]: 100%|██████████| 16/16 [00:06<00:00,  2.39it/s]


[ROARL1] Saving results for german run 0


...

[ROARL1] Saving results for german run 3


[ROARLInf] [alpha=0.1] [lambda=1e-05]: 100%|██████████| 14/14 [00:12<00:00,  1.10it/s]


[ROARLInf] Saving results for german run 4


[ROARL1] [alpha=0.1] [lambda=1e-05]: 100%|██████████| 14/14 [00:06<00:00,  2.24it/s]

[ROARL1] Saving results for german run 4
Finished german

